# Real Estate Data Analysis with Spark

This lab redoes, with Spark, the work of the Seaborn lab, on the same data: the count of transactions per year, the property value per department, then per region and per year, the price per square meter and the correlations. This is on purpose: in each exercise, compare the computing time (`%%time`, on the first line of a cell, measures it) and the memory used (Colab's RAM gauge) with pandas'. pandas loaded every row of the useful columns into memory; Spark only reads the data when an action runs, and only the columns that action needs.

Two exercises change: the value per department is computed with Spark's pandas API, and the value per region and per year caches an intermediate result. Added to them are reading an execution plan and, as an optional exercise, writing a partitioned Parquet file. A last, optional part, for the trainings that covered MLlib, trains a price per square meter model and puts it into production: saving it, batch predictions, then streaming ones.

## Installing PySpark

In [ ]:
!pip install pyspark

## Imports

In [ ]:
import pathlib
import time

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas
import pyspark
import pyspark.sql.functions as F
import seaborn as sns
from pyspark.sql import Column

## Loading the Data

In this lab, we'll use real estate transaction data for all of France, between 2014 and 2022.

This data comes from the [French open data](https://www.data.gouv.fr/fr/datasets/demandes-de-valeurs-foncieres-geolocalisees/) website. We'll use a reworked version that groups transactions, from the [NyxAether/DVF](https://github.com/NyxAether/DVF) repository on GitHub.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh dvf

In Spark SQL, you always start by defining a session, named `spark` by convention. This session specifies how to access the cluster, a few options, etc. Here, we'll use it in its simplest form.

In [ ]:
spark = pyspark.sql.SparkSession.builder.appName("DVF").getOrCreate()

Let's load the data into a Spark dataframe, `df`. `dvf/dvf-linearized-2014-2022.parquet` is a directory, partitioned by year: it holds one subdirectory per value of `annee_mutation` (`annee_mutation=2014`, `annee_mutation=2015`…), from which Spark rebuilds that column. Here, `spark.read.parquet` only reads the list of files and the schema: each column's name and type.

In [ ]:
df = spark.read.parquet("dvf/dvf-linearized-2014-2022.parquet")

Create a `date` column from the `annee_mutation`, `mois_mutation` and `jour_mutation` columns with `F.make_date` and `withColumn`, then drop these columns, which `date` replaces, with `drop`. Display the first ten rows with `show`.

**Useful documentation**

- [`pyspark.sql.functions.make_date`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.make_date.html)
- [`pyspark.sql.DataFrame.withColumn`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.withColumn.html)
- [`pyspark.sql.DataFrame.drop`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.drop.html)
- [`pyspark.sql.DataFrame.show`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.show.html)

In [ ]:
# Your code here

Finally, display the column types of the dataframe with `printSchema`.

**Useful documentation**

- [`pyspark.sql.DataFrame.printSchema`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.printSchema.html)

In [ ]:
# Your code here

### Additional hints

`F.make_date` takes the names of the three columns, in the order year, month, day. `withColumn` returns a new dataframe, with the column added, and `drop` a new dataframe without the named columns: assign the result to `df`. `printSchema` displays each column's name and type: `date` should appear last, with the `date` type.

```python
df = df.withColumn("date", F.make_date(...)).drop(...)
df.show(10)
df.printSchema()
```

### Solution

In [ ]:
df = df.withColumn(
  "date", F.make_date("annee_mutation", "mois_mutation", "jour_mutation")
).drop("annee_mutation", "mois_mutation", "jour_mutation")
df.show(10)

In [ ]:
df.printSchema()

## Lazy Evaluation: the Execution Plan

`withColumn` and `drop` computed nothing: they are transformations, which add a step to `df`'s plan. Only actions, such as `show`, `count` or `toPandas`, start a computation, and `show(10)` only read the first rows.

- Display `df`'s plan with `explain()`. The `FileScan parquet` line gives, between brackets, the columns Spark will read from the file: which ones?
- Display in the same way the plan of the count per year, `df.groupBy(F.year("date")).count()`, without running it: which columns will Spark read this time? Why can it read only those from a Parquet file?
- Count the rows of `df` with `count()` (you should find 10,839,063), then display `df.explain()` again. Has the plan changed? What will Spark do again at the next action on `df`?

**Useful documentation**

- [`pyspark.sql.DataFrame.explain`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.explain.html)
- [`pyspark.sql.DataFrame.count`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.count.html)

In [ ]:
# Your code here

### Additional hints

`df.explain()` displays the physical plan, without computing anything. The line that starts with `FileScan parquet` is the read, and the list between brackets gives the columns read: Spark names 25 of them, then sums up the rest ("... 10 more fields"). The plan of the count is the one of the dataframe that `count()` returns on grouped data, a transformation: `df.groupBy(F.year("date")).count().explain()`. On the whole dataframe, on the other hand, `count()` is an action, which returns an integer.

```python
df.explain()
df.groupBy(...).count().explain()
print(df.count())
...
```

### Solution

In [ ]:
df.explain()
df.groupBy(F.year("date")).count().explain()
print(df.count())
df.explain()
# df's plan reads every column of the file, since it keeps them all. The count's
# plan only reads jour_mutation, mois_mutation and annee_mutation, what date needs:
# Parquet stores each column apart, and annee_mutation comes from the names of the
# subdirectories. count() read the file to compute a number, but df's plan hasn't
# changed: Spark kept nothing, and the next action will read the file again
# (cache(), further down, avoids that).

## Number of Transactions per Year

To start, we'll count the number of transactions (rows of the dataset) per year.

Complete the `get_data_transactions_by_year` function: it returns the number of transactions of each year, sorted by year, in a pandas dataframe. Group the rows by year with `groupBy` and `F.year`, applied to the `date` column, count them with `count`, sort them with `orderBy`, then convert the result with `toPandas`, which returns a data structure seaborn can easily use. Then uncomment the `transactions_by_year()` call, at the bottom of the cell, to see the chart.

The chart creation function expects the data in a pandas dataframe organized like this:

```
   year    count
0  2014  1032967
1  2015  1128516
2  2016  1210587
3  2017  1369101
4  2018  1351856
5  2019  1431358
6  2020  1268193
7  2021  1537648
8  2022   508837
```

**Useful documentation**

- [`pyspark.sql.DataFrame.groupBy`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupBy.html)
- [`pyspark.sql.functions.year`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.year.html)
- [`pyspark.sql.GroupedData.count`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.GroupedData.count.html)
- [`pyspark.sql.DataFrame.orderBy`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.orderBy.html)
- [`pyspark.sql.DataFrame.toPandas`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.toPandas.html)

In [ ]:
def get_data_transactions_by_year() -> pandas.DataFrame:
  pass  # Your code here


def transactions_by_year() -> None:
  data = get_data_transactions_by_year()

  fig, ax = plt.subplots()
  sns.barplot(data, x="year", y="count", ax=ax)
  sns.despine(fig)
  ax.set_title("Transaction count per year")
  ax.set_xlabel("Year")
  ax.set_ylabel("Number of transactions")
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(places=1))
  fig.show()


# transactions_by_year()

### Additional hints

`F.year("date")` gives the year of each date, and `.alias("year")` names this column, which the chart expects under that name. `count()`, called on grouped data, returns a dataframe with a `count` column: the number of rows of each group. Spark doesn't guarantee the order of the groups: sort them by `year`.

```python
def get_data_transactions_by_year() -> pandas.DataFrame:
  return df.groupBy(F.year("date").alias(...)).count().orderBy(...).toPandas()
```

### Solution

In [ ]:
def get_data_transactions_by_year() -> pandas.DataFrame:
  return df.groupBy(F.year("date").alias("year")).count().orderBy("year").toPandas()


def transactions_by_year() -> None:
  data = get_data_transactions_by_year()

  fig, ax = plt.subplots()
  sns.barplot(data, x="year", y="count", ax=ax)
  sns.despine(fig)
  ax.set_title("Transaction count per year")
  ax.set_xlabel("Year")
  ax.set_ylabel("Number of transactions")
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(places=1))
  fig.show()


transactions_by_year()

## Total Property Value by Department, with the pandas API

In this exercise, we'll group our data by the values of a column (here the department of the transaction) and display the total property value exchanged per group, this time with Spark's pandas API: pandas code, which Spark runs.

With Spark 4.0, Colab's, the pandas API refuses to start in ANSI mode, which Spark 4 turns on by default: the exercise's cell starts by turning it off (this is no longer needed since Spark 4.1).

Complete the `get_data_value_by_department` function, which returns a pandas series:

- Switch to the pandas API with `pandas_api`, which returns a pandas-on-Spark dataframe.
- Group it by department (`code_departement`) and compute the sum of `valeur_fonciere` for each group with `groupby` and `sum`, then sort the sums in decreasing order with `sort_values`, as with pandas.
- Convert the result into a pandas series with `to_pandas`.

Compare this code with the Seaborn lab's for the same chart. Then uncomment the `value_by_department()` call to see the chart.

The expected pandas series looks like this:

```
code_departement
75    1.821853e+11
92    1.327795e+11
6     8.143718e+10
33    8.015320e+10
69    7.632324e+10
          ...
55    1.998241e+09
15    1.907638e+09
52    1.753427e+09
23    1.248303e+09
48    8.985644e+08
Name: valeur_fonciere, Length: 97, dtype: float64
```

**Useful documentation**

- [`pyspark.sql.DataFrame.pandas_api`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.pandas_api.html)
- [`pyspark.pandas.DataFrame.groupby`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.pandas/api/pyspark.pandas.DataFrame.groupby.html)
- [`pyspark.pandas.Series.sort_values`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.pandas/api/pyspark.pandas.Series.sort_values.html)
- [`pyspark.pandas.Series.to_pandas`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.pandas/api/pyspark.pandas.Series.to_pandas.html)

In [ ]:
# Spark 4.0 (Colab): the pandas API refuses to start in ANSI mode
spark.conf.set("spark.sql.ansi.enabled", "false")


def get_data_value_by_department() -> pandas.Series:
  pass  # Your code here


def value_by_department() -> None:
  data = get_data_value_by_department()

  fig, ax = plt.subplots(figsize=(30, 10))
  sns.barplot(x=data.index, y=data, order=data.index, ax=ax)
  sns.despine(ax=ax)
  ax.set_title("Total transaction amount per department")
  ax.set_xlabel("Department")
  ax.set_ylabel("Total transaction amount")
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(unit="€", places=1))
  fig.show()


# value_by_department()

### Additional hints

`df.pandas_api()` returns a pandas-on-Spark dataframe, whose methods are pandas': `groupby` (lowercase, as in pandas, not `groupBy`), selecting a column by its attribute name, `sum`, `sort_values(ascending=False)`. The result is still a pandas-on-Spark series, which Spark hasn't computed yet: `to_pandas()` starts the computation and brings the 97 sums back in a pandas series.

```python
def get_data_value_by_department() -> pandas.Series:
  df_pandas = df.pandas_api()
  return df_pandas.groupby(...).valeur_fonciere.sum().sort_values(...).to_pandas()
```

### Solution

In [ ]:
# Spark 4.0 (Colab): the pandas API refuses to start in ANSI mode
spark.conf.set("spark.sql.ansi.enabled", "false")


def get_data_value_by_department() -> pandas.Series:
  df_pandas = df.pandas_api()
  return (
    df_pandas.groupby("code_departement")
    .valeur_fonciere.sum()
    .sort_values(ascending=False)
    .to_pandas()
  )


def value_by_department() -> None:
  data = get_data_value_by_department()

  fig, ax = plt.subplots(figsize=(30, 10))
  sns.barplot(x=data.index, y=data, order=data.index, ax=ax)
  sns.despine(ax=ax)
  ax.set_title("Total transaction amount per department")
  ax.set_xlabel("Department")
  ax.set_ylabel("Total transaction amount")
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(unit="€", places=1))
  fig.show()


value_by_department()

## Total Property Value by Region and by Year

In this part, we'll study how to break down a categorical variable by another variable: we'll produce a chart of the total property value exchanged by region, and this value will itself be broken down by year.

To do so, let's start by fetching a dataset that will let us group departments into regions:

In [ ]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/france-departements/departements-region.csv -O regions.csv

The following function returns a Spark dataframe that makes it easy to join the region to our working dataframe: its `num_dep` column is the department's code, written as in our data ("1", not "01"), and its `region_name` column the region's name.

In [ ]:
def get_regions_df() -> pyspark.sql.DataFrame:
  return (
    spark.read.csv("regions.csv", inferSchema=True, header=True)
    .drop("dep_name")
    .withColumn("num_dep", F.regexp_replace("num_dep", "^0+", ""))
  )


get_regions_df().show(10)

Complete the `get_data_value_by_region_by_year` function, which returns a pandas dataframe and a pandas series:

- Add a column containing the region name to your working dataframe with `join`, matching `code_departement` with `num_dep`, and keep, with `select`, only the three useful columns: the region, the year of the date, under the name `year`, and the property value.
- This dataframe serves two aggregations, and Spark would redo the read and the join for each: cache it with `cache`, before both.
- Return a pandas dataframe containing the sum of the property value exchanged per region and per year, under the name `total`, with `groupBy` on two columns.
- Also return a pandas series: the region names sorted from the region with the largest sum of property value exchanged to the smallest.
- Measure the duration of each of the two aggregations with `time.perf_counter`, then free the cache with `unpersist`.

Then run the function again without `cache()`, and compare the durations: which aggregation does the cache speed up, which one does it slow down, and why? When is it worth it?

Then uncomment the `value_by_region_by_year()` call to see the chart.

The dataframe to return looks like this (the order of its rows can change from one run to the next):

```
                 region_name  year         total
0              Île-de-France  2021  9.168523e+10
1    Bourgogne-Franche-Comté  2021  8.240557e+09
2                 La Réunion  2021  1.681132e+09
3                     Guyane  2021  5.290445e+08
4                   Bretagne  2021  1.613739e+10
..                       ...   ...           ...
148                Occitanie  2022  8.358505e+09
149               Guadeloupe  2022  5.992970e+07
150               La Réunion  2022  2.251319e+08
151                 Bretagne  2022  4.873580e+09
152                Normandie  2022  2.027011e+09
```

The series to return looks like this:

```
0                  Île-de-France
1           Auvergne-Rhône-Alpes
2     Provence-Alpes-Côte d'Azur
3             Nouvelle-Aquitaine
4                      Occitanie
5                Hauts-de-France
6               Pays de la Loire
7                       Bretagne
8                      Normandie
9            Centre-Val de Loire
10       Bourgogne-Franche-Comté
11                     Grand Est
12                    La Réunion
13                         Corse
14                    Guadeloupe
15                    Martinique
16                        Guyane
Name: region_name, dtype: object
```

**Useful documentation**

- [`pyspark.sql.DataFrame.join`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html)
- [`pyspark.sql.DataFrame.select`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.select.html)
- [`pyspark.sql.DataFrame.cache`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.cache.html)
- [`pyspark.sql.DataFrame.unpersist`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.unpersist.html)
- [`time.perf_counter`](https://docs.python.org/3/library/time.html#time.perf_counter)

In [ ]:
def get_data_value_by_region_by_year() -> tuple[pandas.DataFrame, pandas.Series]:
  pass  # Your code here


def value_by_region_by_year() -> None:
  data, sorted_index = get_data_value_by_region_by_year()

  fig, ax = plt.subplots(figsize=(20, 10))
  sns.barplot(data, x="region_name", y="total", hue="year", order=sorted_index, ax=ax)
  sns.despine(fig)
  ax.set_title("Total transaction amount by region and by year")
  ax.set_xlabel("Region")
  plt.xticks(rotation=60)
  ax.set_ylabel("Total transaction amount")
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(unit="€", places=1))
  fig.show()


# value_by_region_by_year()

### Additional hints

`df.join(regions_df, df.code_departement == regions_df.num_dep)` keeps the rows whose department has a region and adds the columns of `regions_df` to them. `select` takes column names or expressions, such as `F.year("date").alias("year")`. `cache()` returns the same dataframe, marked to be kept in memory: Spark computes and stores it at the first action that uses it, then reads it from there. Keep only the three columns: without `select`, the cache would keep every column of the file. `groupBy` accepts several columns. For the series, group by region alone, sort the sums in decreasing order, keep the `region_name` column with `select`, then take it as a series after `toPandas()`, with `.iloc[:, 0]`. `time.perf_counter()` returns a point in time in seconds: the duration is the difference of two calls. The function returns both results in a tuple: `return data, sorted_index`.

```python
def get_data_value_by_region_by_year() -> tuple[pandas.DataFrame, pandas.Series]:
  regions_df = get_regions_df()
  joined = (
    df.join(regions_df, ...).select("region_name", ..., "valeur_fonciere").cache()
  )
  start = time.perf_counter()
  data = joined.groupBy(...).agg(...).toPandas()
  print(f"By region and by year: {time.perf_counter() - start:.1f} s")
  start = time.perf_counter()
  sorted_index = (
    joined.groupBy(...)
    .agg(...)
    .orderBy(..., ascending=False)
    .select("region_name")
    .toPandas()
    .iloc[:, 0]
  )
  print(f"By region: {time.perf_counter() - start:.1f} s")
  joined.unpersist()
  return data, sorted_index
```

### Solution

In [ ]:
def get_data_value_by_region_by_year() -> tuple[pandas.DataFrame, pandas.Series]:
  regions_df = get_regions_df()
  joined = (
    df.join(regions_df, df.code_departement == regions_df.num_dep)
    .select("region_name", F.year("date").alias("year"), "valeur_fonciere")
    .cache()
  )
  start = time.perf_counter()
  data = (
    joined.groupBy("region_name", "year")
    .agg(F.sum("valeur_fonciere").alias("total"))
    .toPandas()
  )
  print(f"By region and by year: {time.perf_counter() - start:.1f} s")
  start = time.perf_counter()
  sorted_index = (
    joined.groupBy("region_name")
    .agg(F.sum("valeur_fonciere").alias("total"))
    .orderBy("total", ascending=False)
    .select("region_name")
    .toPandas()
    .iloc[:, 0]
  )
  print(f"By region: {time.perf_counter() - start:.1f} s")
  joined.unpersist()
  # The first aggregation computes the cache and stores it: it is two to three times
  # slower than without the cache. The second one reads it: it is twice as fast.
  # Overall, the cache loses here, since reading three Parquet columns again and
  # joining a small table is cheap. It pays off when a dataframe serves often, or is
  # costly to recompute (a big join, a CSV file to parse).
  return data, sorted_index


def value_by_region_by_year() -> None:
  data, sorted_index = get_data_value_by_region_by_year()

  fig, ax = plt.subplots(figsize=(20, 10))
  sns.barplot(data, x="region_name", y="total", hue="year", order=sorted_index, ax=ax)
  sns.despine(fig)
  ax.set_title("Total transaction amount by region and by year")
  ax.set_xlabel("Region")
  plt.xticks(rotation=60)
  ax.set_ylabel("Total transaction amount")
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(unit="€", places=1))
  fig.show()


value_by_region_by_year()

## House and Apartment Prices

In this exercise, we'll compute the price per square meter of apartments and houses separately, and display how these prices evolve over the period considered.

Complete the `get_data_houses_apartments_prices` function:

- For apartments, filter the working dataframe with `filter` to keep only the rows where:
    - `nombre_appartements`, `valeur_fonciere` and `surface_reelle_bati_appartements` are strictly positive;
    - `nombre_maisons` is `0`.
- Likewise, for houses, filter the working dataframe to keep only the rows where:
    - `nombre_maisons`, `valeur_fonciere` and `surface_reelle_bati_maisons` are strictly positive;
    - `nombre_appartements` is `0`.
- Use this filtered data to compute the price per square meter of each month of the dataset, for houses and apartments separately: the sum of the month's property values divided by the sum of the areas. `F.date_format` gives a column containing the date in the `yyyy-MM` form, by which to group the rows with `groupBy`.
- Bring the apartments' prices and the houses' together with `join`, on the month. The chart function used, `seaborn.lineplot`, expects data in long form: `melt` provides it.

Then uncomment the `houses_apartments_prices()` call to see the chart.

The expected pandas dataframe looks like this:

```
       month       type        price
0    2014-01  apartment  3005.224170
1    2014-01      house  1588.628790
2    2014-02  apartment  2878.714589
3    2014-02      house  1622.620962
4    2014-03  apartment  2911.870507
..       ...        ...          ...
199  2022-04      house  2190.928353
200  2022-05  apartment  4164.208224
201  2022-05      house  2150.579262
202  2022-06  apartment  4530.162208
203  2022-06      house  2242.808145
```

**Useful documentation**

- [`pyspark.sql.DataFrame.filter`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.filter.html)
- [`pyspark.sql.functions.date_format`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.date_format.html)
- [`pyspark.sql.functions.col`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.col.html)
- [`pyspark.sql.DataFrame.melt`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.melt.html)
- [`seaborn.lineplot`](https://seaborn.pydata.org/generated/seaborn.lineplot.html)

In [ ]:
def get_data_houses_apartments_prices() -> pandas.DataFrame:
  pass  # Your code here


def houses_apartments_prices() -> None:
  data = get_data_houses_apartments_prices()

  fig, ax = plt.subplots()
  sns.lineplot(data, x="month", y="price", hue="type", legend=True)
  plt.xticks(rotation=45)
  ax.set_xticks(ax.get_xticks()[::12])
  ax.set_xlabel("Date")
  ax.set_ylabel("Average value")
  ax.set_title("Price per square meter")
  ax.set_ylim((0, data.select_dtypes("number").max().max()))
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(unit="€/m²", places=1))
  fig.show()


# houses_apartments_prices()

### Additional hints

A condition on a column, such as `df.nombre_appartements > 0`, is a `Column` of booleans, which `&` combines with another, each condition in parentheses. An intermediate function, called once per type of property, avoids writing the computation twice: it takes the condition, the name of the area column and that of the price column (`apartment` or `house`). In `agg`, name the two sums with `.alias(...)` (`value_sum`, `area_sum`) to divide one by the other afterwards with `F.col`. `join(other, "month")` brings two dataframes together on their common column, and `melt(ids="month", values=["apartment", "house"], variableColumnName="type", valueColumnName="price")` puts them in long form, before `toPandas()`.

```python
def get_data_houses_apartments_prices() -> pandas.DataFrame:
  mask_apartments = (df.nombre_appartements > 0) & (df.nombre_maisons == 0) & ...
  mask_houses = ...

  def get_prices(
    mask: Column, area_column: str, column_name: str
  ) -> pyspark.sql.DataFrame:
    return (
      df.filter(mask)
      .groupBy(F.date_format("date", "yyyy-MM").alias("month"))
      .agg(
        F.sum("valeur_fonciere").alias("value_sum"),
        F.sum(...).alias("area_sum"),
      )
      .withColumn(column_name, ...)
      .drop("value_sum", "area_sum")
    )

  prices_apartments = get_prices(
    mask_apartments, "surface_reelle_bati_appartements", "apartment"
  )
  prices_houses = get_prices(...)

  return (
    prices_apartments.join(prices_houses, "month").orderBy("month").melt(...).toPandas()
  )
```

### Solution

In [ ]:
def get_data_houses_apartments_prices() -> pandas.DataFrame:
  mask_apartments = (
    (df.nombre_appartements > 0)
    & (df.nombre_maisons == 0)
    & (df.valeur_fonciere > 0)
    & (df.surface_reelle_bati_appartements > 0)
  )
  mask_houses = (
    (df.nombre_appartements == 0)
    & (df.nombre_maisons > 0)
    & (df.valeur_fonciere > 0)
    & (df.surface_reelle_bati_maisons > 0)
  )

  def get_prices(
    mask: Column, area_column: str, column_name: str
  ) -> pyspark.sql.DataFrame:
    return (
      df.filter(mask)
      .groupBy(F.date_format("date", "yyyy-MM").alias("month"))
      .agg(
        F.sum("valeur_fonciere").alias("value_sum"),
        F.sum(area_column).alias("area_sum"),
      )
      .withColumn(column_name, F.col("value_sum") / F.col("area_sum"))
      .drop("value_sum", "area_sum")
    )

  prices_apartments = get_prices(
    mask_apartments, "surface_reelle_bati_appartements", "apartment"
  )
  prices_houses = get_prices(mask_houses, "surface_reelle_bati_maisons", "house")

  return (
    prices_apartments.join(prices_houses, "month")
    .orderBy("month")
    .melt(
      ids="month",
      values=["apartment", "house"],
      variableColumnName="type",
      valueColumnName="price",
    )
    .toPandas()
  )


def houses_apartments_prices() -> None:
  data = get_data_houses_apartments_prices()

  fig, ax = plt.subplots()
  sns.lineplot(data, x="month", y="price", hue="type", legend=True)
  plt.xticks(rotation=45)
  ax.set_xticks(ax.get_xticks()[::12])
  ax.set_xlabel("Date")
  ax.set_ylabel("Average value")
  ax.set_title("Price per square meter")
  ax.set_ylim((0, data.select_dtypes("number").max().max()))
  ax.get_yaxis().set_major_formatter(mpl.ticker.EngFormatter(unit="€/m²", places=1))
  fig.show()


houses_apartments_prices()

## Pearson and Spearman Correlations

As in the Seaborn lab, let's compare the Pearson and Spearman correlations, here on the sales of apartments alone in 2021, filtered as in the previous exercise: at least one apartment and no house, a strictly positive property value and area. The variables (the `variables` list) are quantities: neither the postcode, nor the street number, nor the coordinates, which are identifiers.

- Keep these sales and these columns with `filter` and `select`, switch to the pandas API with `pandas_api`, then compute both correlation matrices with `corr` and its `method` argument.
- Display them side by side, on the two `Axes` of one figure, with `sns.heatmap`, each cell annotated with its value.
- Read the `valeur_fonciere` row: with `surface_reelle_bati_appartements`, you should find 0.35 (Pearson) and 0.41 (Spearman); with `nombre_appartements`, 0.25 and 0.06.

Why is the number of apartments of a sale much more correlated to its value according to Pearson than according to Spearman? And why does the area explain only part of the value, even for apartments alone?

**Useful documentation**

- [`pyspark.pandas.DataFrame.corr`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.pandas/api/pyspark.pandas.DataFrame.corr.html)
- [`seaborn.heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html)

In [ ]:
variables = [
  "valeur_fonciere",
  "surface_reelle_bati_appartements",
  "nombre_pieces_principales",
  "nombre_appartements",
  "nombre_lots",
]


def correlations() -> None:
  pass  # Your code here


correlations()

### Additional hints

The condition is the one of the apartments in the previous exercise, plus `F.year("date") == 2021`, and `select(variables)` keeps the columns of the list. On the pandas-on-Spark dataframe (ANSI mode is turned off above), `corr(method="spearman")` computes the Spearman correlation (`"pearson"` by default) and returns a small pandas-on-Spark dataframe, which `to_pandas()` converts for seaborn. `fig, axes = plt.subplots(1, 2, figsize=(18, 7))` creates a figure with two `Axes`, `axes[0]` and `axes[1]`, which `sns.heatmap` takes through `ax`. `annot=True` writes the value in each cell, `fmt=".2f"` with two decimals; `vmin=-1, vmax=1` set the color scale, and a diverging palette (`cmap="vlag"`) tells its sign apart.

```python
def correlations() -> None:
  apartments_2021 = (
    df.filter((F.year("date") == 2021) & (df.nombre_appartements > 0) & ...)
    .select(variables)
    .pandas_api()
  )
  fig, axes = plt.subplots(1, 2, figsize=(18, 7))
  for ax, method in zip(axes, ["pearson", "spearman"]):
    sns.heatmap(
      apartments_2021.corr(method=...).to_pandas(),
      vmin=-1,
      vmax=1,
      cmap="vlag",
      annot=...,
      fmt=".2f",
      ax=ax,
    )
    ax.set_title(f"Correlations ({method}, apartments, 2021)")
  fig.tight_layout()
  fig.show()
```

### Solution

In [ ]:
def correlations() -> None:
  apartments_2021 = (
    df.filter(
      (F.year("date") == 2021)
      & (df.nombre_appartements > 0)
      & (df.nombre_maisons == 0)
      & (df.valeur_fonciere > 0)
      & (df.surface_reelle_bati_appartements > 0)
    )
    .select(variables)
    .pandas_api()
  )
  fig, axes = plt.subplots(1, 2, figsize=(18, 7))
  for ax, method in zip(axes, ["pearson", "spearman"]):
    sns.heatmap(
      apartments_2021.corr(method=method).to_pandas(),
      vmin=-1,
      vmax=1,
      cmap="vlag",
      annot=True,
      fmt=".2f",
      ax=ax,
    )
    ax.set_title(f"Correlations ({method}, apartments, 2021)")
  fig.tight_layout()
  fig.show()
  # 93% of these sales are of a single apartment: Spearman, which only sees ranks,
  # barely links their number to the value (0.06). Pearson (0.25) follows the few
  # bulk sales, of several apartments and of high value, which dominate the
  # variance. The area explains only part of the value (0.35 and 0.41), since the
  # price per square meter depends on the place: in 2021, its median goes from
  # 580 €/m² in the Creuse to 10,700 €/m² in Paris, where Spearman rises to 0.83.


correlations()

## Writing Parquet Partitioned by Year (Optional)

The DVF file is partitioned by year: one subdirectory per value of `annee_mutation`. Let's write the sales in the same way, cut down to a few columns, and see what Spark gains from it when reading.

- Keep the `date`, `code_departement` and `valeur_fonciere` columns of `df`, add a `year` column to them (`F.year`), then write the whole as Parquet into `sales-by-year.parquet`, partitioned by `year`, with `write`, `partitionBy` and `parquet` (`mode("overwrite")` lets you run the cell again).
- List the content of the written directory with `pathlib.Path.iterdir`.
- Read it back with `spark.read.parquet`, keep the sales of 2021 with `filter`, and display their plan with `explain()`: what does the `PartitionFilters` line say? Count them: you should find 1,537,648 sales.

**Useful documentation**

- [`pyspark.sql.DataFrame.write`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.write.html)
- [`pyspark.sql.DataFrameWriter.partitionBy`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html)
- [`pyspark.sql.DataFrameWriter.mode`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.mode.html)
- [`pyspark.sql.DataFrameWriter.parquet`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.parquet.html)
- [`pathlib.Path.iterdir`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.iterdir)

In [ ]:
# Your code here

### Additional hints

`df.write` returns a `DataFrameWriter`, whose methods chain: `partitionBy("year")` gives the column that names the subdirectories, `mode("overwrite")` replaces a directory already written, and `parquet(path)` starts the write (an action). `sorted(p.name for p in pathlib.Path(path).iterdir())` lists the `year=2014`… subdirectories and the `_SUCCESS` file, which marks a finished write. In the dataframe read back, `year` is a column again: a filter on it shows up in `PartitionFilters`.

```python
(
  df.select(...)
  .withColumn("year", ...)
  .write.partitionBy(...)
  .mode("overwrite")
  .parquet("sales-by-year.parquet")
)
print(sorted(p.name for p in pathlib.Path("sales-by-year.parquet").iterdir()))
sales = spark.read.parquet(...)
sales_2021 = sales.filter(...)
sales_2021.explain()
print(sales_2021.count())
```

### Solution

In [ ]:
(
  df.select("date", "code_departement", "valeur_fonciere")
  .withColumn("year", F.year("date"))
  .write.partitionBy("year")
  .mode("overwrite")
  .parquet("sales-by-year.parquet")
)
print(sorted(p.name for p in pathlib.Path("sales-by-year.parquet").iterdir()))
sales = spark.read.parquet("sales-by-year.parquet")
sales_2021 = sales.filter(sales.year == 2021)
sales_2021.explain()
print(sales_2021.count())
# PartitionFilters: [isnotnull(year), (year = 2021)]: Spark only reads the
# year=2021 subdirectory, without opening those of the other years.

## A Price per Square Meter Model (Optional)

This part, for the trainings that covered MLlib, moves from analysis to prediction: a model that estimates the price per square meter of an apartment from its location, its area and its number of rooms, fitted on the sales of 2021. It follows that model up to its production use: a `Pipeline` fitted and evaluated, saved, reloaded as another program would, then applied to the sales of 2022, in batch, then as a stream.

The directories these exercises write (the model, the predictions, the stream's checkpoint) are relative to the current directory, like the partitioned Parquet above, and each cell replaces them: you can run it again.

In [ ]:
import shutil

from pyspark.ml import Estimator, Pipeline, PipelineModel
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GBTRegressor, LinearRegression

### Apartment Sales

Complete the `apartments` function, which prepares the apartment sales of a dataframe of sales, for training as well as for predictions:

- filter them as in the "Correlations" part: at least one apartment and no house, a strictly positive property value and area;
- add the target to them, the price per square meter, in a `price_m2` column;
- keep the prices between 500 and 20,000 €/m², with `between`: outside of them are mostly transfers at a symbolic price, or sales whose value covers more than the apartments (2% of the sales of 2021);
- finally, keep the date, the columns of the `feature_columns` list and `price_m2`.

Apply it to the sales of 2021, then split the result into a training set (80%) and a test set (20%) with `randomSplit`. The next exercises use them several times: cache them, and count their rows. You should find about 353,000 and 89,000 sales.

**Useful documentation**

- [`pyspark.sql.Column.between`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Column.between.html)
- [`pyspark.sql.DataFrame.randomSplit`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.randomSplit.html)

In [ ]:
feature_columns = [
  "longitude",
  "latitude",
  "surface_reelle_bati_appartements",
  "nombre_pieces_principales",
]


def apartments(sales: pyspark.sql.DataFrame) -> pyspark.sql.DataFrame:
  pass  # Your code here


# Your code here

#### Additional hints

The condition is the apartments' one of the "Correlations" part, written on `sales` rather than on `df`. `sales.valeur_fonciere / sales.surface_reelle_bati_appartements` divides two columns row by row, and `F.col("price_m2").between(500, 20000)` is true for the prices between the two bounds, included. `select("date", *feature_columns, "price_m2")` unpacks the list into column names. `randomSplit([0.8, 0.2], seed=0)` returns a list of two dataframes, to unpack: `seed` makes the split reproducible, and gives the numbers of the statement. `cache()` returns the marked dataframe, which `count()` computes and stores.

```python
def apartments(sales: pyspark.sql.DataFrame) -> pyspark.sql.DataFrame:
  return (
    sales.filter((sales.nombre_appartements > 0) & (sales.nombre_maisons == 0) & ...)
    .withColumn("price_m2", ...)
    .filter(...)
    .select(...)
  )


train, test = apartments(df.filter(...)).randomSplit(...)
train = train.cache()
test = test.cache()
print(train.count(), test.count())
```

#### Solution

In [ ]:
feature_columns = [
  "longitude",
  "latitude",
  "surface_reelle_bati_appartements",
  "nombre_pieces_principales",
]


def apartments(sales: pyspark.sql.DataFrame) -> pyspark.sql.DataFrame:
  return (
    sales.filter(
      (sales.nombre_appartements > 0)
      & (sales.nombre_maisons == 0)
      & (sales.valeur_fonciere > 0)
      & (sales.surface_reelle_bati_appartements > 0)
    )
    .withColumn(
      "price_m2", sales.valeur_fonciere / sales.surface_reelle_bati_appartements
    )
    .filter(F.col("price_m2").between(500, 20000))
    .select("date", *feature_columns, "price_m2")
  )


train, test = apartments(df.filter(F.year("date") == 2021)).randomSplit(
  [0.8, 0.2], seed=0
)
train = train.cache()
test = test.cache()
print(train.count(), test.count())

### One Pipeline, Two Models

A `Pipeline` chains stages: here a `VectorAssembler`, which gathers the `feature_columns` into a vector, in the `features` column that MLlib's models expect, then a regressor, which predicts `price_m2`. Its `fit` fits the stages on the training set and returns a `PipelineModel`, whose `transform` adds the `prediction` column to the data.

- Complete the `build_pipeline` function, which returns the pipeline of a given regressor.
- Complete the `evaluate` function, which displays two metrics of a model's predictions with `RegressionEvaluator`: the root mean squared error (`rmse`, in €/m²) and the coefficient of determination (`r2`).
- Fit the pipeline of a `LinearRegression`, then evaluate the model on the test set.
- Do it again with a `GBTRegressor` (gradient-boosted decision trees) of 10 trees, in `model`, and measure the duration of its `fit` with `time.perf_counter`.

You should find an R² close to 0 for the linear regression, and of about 0.68 for the trees, whose RMSE is about 1,600 €/m². Why does a linear model explain almost nothing of the price with these features? What do the trees gain?

**Useful documentation**

- [`pyspark.ml.Pipeline`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.Pipeline.html)
- [`pyspark.ml.feature.VectorAssembler`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.VectorAssembler.html)
- [`pyspark.ml.regression.LinearRegression`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.regression.LinearRegression.html)
- [`pyspark.ml.regression.GBTRegressor`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.regression.GBTRegressor.html)
- [`pyspark.ml.evaluation.RegressionEvaluator`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.evaluation.RegressionEvaluator.html)

In [ ]:
def build_pipeline(regressor: Estimator) -> Pipeline:
  pass  # Your code here


def evaluate(model: PipelineModel, data: pyspark.sql.DataFrame) -> None:
  pass  # Your code here


# Your code here

#### Additional hints

`Pipeline(stages=[...])` takes the list of the stages, in order. `VectorAssembler(inputCols=feature_columns, outputCol="features")` assembles the columns, and a regressor names its target with `labelCol="price_m2"` (its input column is `features` by default). `GBTRegressor` takes the number of trees with `maxIter=10`, and `seed=0` makes the fit reproducible. `RegressionEvaluator(labelCol="price_m2", metricName="r2")` compares `price_m2` with the `prediction` column, and its `evaluate` takes the data transformed by the model.

```python
def build_pipeline(regressor: Estimator) -> Pipeline:
  return Pipeline(stages=[VectorAssembler(...), regressor])


def evaluate(model: PipelineModel, data: pyspark.sql.DataFrame) -> None:
  predictions = model.transform(data)
  for metric in ["rmse", "r2"]:
    evaluator = RegressionEvaluator(...)
    print(f"{metric}: {evaluator.evaluate(predictions):.2f}")


linear_model = build_pipeline(LinearRegression(...)).fit(train)
evaluate(linear_model, test)
start = time.perf_counter()
model = build_pipeline(GBTRegressor(...)).fit(...)
print(f"Fit: {time.perf_counter() - start:.0f} s")
evaluate(...)
```

#### Solution

In [ ]:
def build_pipeline(regressor: Estimator) -> Pipeline:
  return Pipeline(
    stages=[
      VectorAssembler(inputCols=feature_columns, outputCol="features"),
      regressor,
    ]
  )


def evaluate(model: PipelineModel, data: pyspark.sql.DataFrame) -> None:
  predictions = model.transform(data)
  for metric in ["rmse", "r2"]:
    evaluator = RegressionEvaluator(labelCol="price_m2", metricName=metric)
    print(f"{metric}: {evaluator.evaluate(predictions):.2f}")


linear_model = build_pipeline(LinearRegression(labelCol="price_m2")).fit(train)
evaluate(linear_model, test)
start = time.perf_counter()
model = build_pipeline(GBTRegressor(labelCol="price_m2", maxIter=10, seed=0)).fit(train)
print(f"Fit: {time.perf_counter() - start:.0f} s")
evaluate(model, test)
# The price per square meter depends on the location, but not along a straight line:
# a linear function of the longitude and the latitude only describes a slope (more
# expensive in the north, or in the east), while prices peak in Paris, on the French
# Riviera or in the Alps. The trees cut the map into areas, each with its own price,
# and cross the location with the area.

### Saving and Reloading the Model

A model is used elsewhere than where it was fitted: in a batch job, every night, or in a service.

- Save `model` into the `price-m2-model` directory with `write().overwrite().save(...)`, then list the content of its `stages` subdirectory with `pathlib.Path.iterdir`: what do you find there?
- Reload it into `reloaded_model` with `PipelineModel.load`, as another program would, and check with `evaluate` that it gives the same metrics on the test set.

**Useful documentation**

- [`pyspark.ml.PipelineModel`](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.PipelineModel.html)

In [ ]:
# Your code here

#### Additional hints

`model.write()` returns a writer, whose `overwrite()` lets it replace a directory already written, and `save(path)` writes the model. `PipelineModel.load(path)` is a method of the class: it needs no existing object, only the directory.

```python
model.write().overwrite().save("price-m2-model")
print(sorted(p.name for p in pathlib.Path("price-m2-model/stages").iterdir()))
reloaded_model = PipelineModel.load(...)
evaluate(...)
```

#### Solution

In [ ]:
model.write().overwrite().save("price-m2-model")
print(sorted(p.name for p in pathlib.Path("price-m2-model/stages").iterdir()))
reloaded_model = PipelineModel.load("price-m2-model")
evaluate(reloaded_model, test)
# stages holds one directory per stage, numbered in the pipeline's order: the
# VectorAssembler, with its columns, and the fitted model, with its trees. The
# reloaded PipelineModel redoes the whole preparation: the program that uses it only
# needs the raw columns.

### Batch Predictions for One Month of 2022

A batch job applies the reloaded model to a whole batch of data: here the apartment sales of June 2022, the last month of the data, which the model, fitted on 2021, has never seen.

- Keep the sales of June 2022 from `df` with `F.year` and `F.month`, prepare them with `apartments`, then apply `reloaded_model.transform` to them.
- Write the predictions, without the `features` column, which their readers don't need, as Parquet into `predictions-2022-06.parquet` (`mode("overwrite")`).
- Read them back, count them (you should find 25,181 sales), and compare the mean of `price_m2` with that of `prediction` with `agg` and `F.avg`. Also evaluate the model on these sales with `evaluate`: its RMSE goes from about 1,600 to about 1,800 €/m².

Does the model underestimate or overestimate the prices of 2022? Why, and what to do about it in production? Here, the true price of each sale is known, because DVF publishes past sales: in production, it isn't when the prediction is made.

**Useful documentation**

- [`pyspark.sql.functions.month`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.month.html)
- [`pyspark.sql.DataFrame.agg`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.agg.html)
- [`pyspark.sql.functions.avg`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.avg.html)

In [ ]:
# Your code here

#### Additional hints

The two conditions on the date combine with `&`, each one in parentheses. `transform` returns a dataframe, which Spark only computes when writing: `drop("features")`, then `write.mode("overwrite").parquet(path)`, as for the partitioned Parquet. `agg(F.avg("price_m2"), F.avg("prediction"))` computes the two means in one pass, and `show()` displays them.

```python
june_2022 = apartments(df.filter((F.year("date") == 2022) & ...))
(
  reloaded_model.transform(june_2022)
  .drop(...)
  .write.mode("overwrite")
  .parquet("predictions-2022-06.parquet")
)
june_predictions = spark.read.parquet(...)
print(june_predictions.count())
june_predictions.agg(...).show()
evaluate(reloaded_model, june_2022)
```

#### Solution

In [ ]:
june_2022 = apartments(df.filter((F.year("date") == 2022) & (F.month("date") == 6)))
(
  reloaded_model.transform(june_2022)
  .drop("features")
  .write.mode("overwrite")
  .parquet("predictions-2022-06.parquet")
)
june_predictions = spark.read.parquet("predictions-2022-06.parquet")
print(june_predictions.count())
june_predictions.agg(F.avg("price_m2"), F.avg("prediction")).show()
evaluate(reloaded_model, june_2022)
# The model underestimates the prices of June 2022 by about 5%: prices have risen
# since 2021, and it only knows those of 2021. This is a drift, which the monitoring
# of a model in production detects (the mean of the predictions, then the error once
# the true prices are known), and which regular refitting on recent sales fixes.

### Streaming Predictions (Optional)

In a stream, Spark processes the data as it arrives. Structured Streaming reads a directory as a table that grows, and only processes the new files at each pass, thanks to its checkpoint, a directory where it notes what it has already processed.

The exercise's cell simulates the arrival of the sales: `deliver(month)` writes the sales of one month of 2022 into the `incoming` directory, partitioned by month, and the cell delivers January and February there. The schema of these sales is in `sales_schema`.

- Complete the `predict_stream` function: read `incoming` as a stream with `spark.readStream`, giving it the schema (a stream doesn't infer it), prepare the sales with `apartments`, apply `reloaded_model.transform` to them, the same code as in batch, then write the predictions, without `features`, as Parquet into `predictions-stream`, with `writeStream`. Give it the `predictions-stream-checkpoint` checkpoint and `trigger(availableNow=True)`, which processes everything that has arrived then stops, and wait for the end with `awaitTermination`.
- Call it, then count the predictions written (`spark.read.parquet`). Deliver March with `deliver(3)`, call it again and count again: what did the second pass read? You should find 52,041, then 82,293 predictions.

**Useful documentation**

- [`pyspark.sql.SparkSession.readStream`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.readStream.html)
- [`pyspark.sql.streaming.DataStreamReader.schema`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.DataStreamReader.schema.html)
- [`pyspark.sql.streaming.DataStreamReader.parquet`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.DataStreamReader.parquet.html)
- [`pyspark.sql.DataFrame.writeStream`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.writeStream.html)
- [`pyspark.sql.streaming.DataStreamWriter.option`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.DataStreamWriter.option.html)
- [`pyspark.sql.streaming.DataStreamWriter.trigger`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.DataStreamWriter.trigger.html)
- [`pyspark.sql.streaming.DataStreamWriter.start`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.DataStreamWriter.start.html)
- [`pyspark.sql.streaming.StreamingQuery.awaitTermination`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.StreamingQuery.awaitTermination.html)

In [ ]:
# Start from scratch at each run: no sales delivered, no predictions, no checkpoint
for directory in ["incoming", "predictions-stream", "predictions-stream-checkpoint"]:
  shutil.rmtree(directory, ignore_errors=True)
sales_2022 = df.filter(F.year("date") == 2022).cache()


def deliver(month: int) -> None:
  (
    sales_2022.filter(F.month("date") == month)
    .withColumn("month", F.lit(month))
    .write.partitionBy("month")
    .mode("append")
    .parquet("incoming")
  )


deliver(1)
deliver(2)
sales_schema = spark.read.parquet("incoming").schema


def predict_stream() -> None:
  pass  # Your code here


# Your code here

#### Additional hints

`spark.readStream.schema(sales_schema).parquet("incoming")` returns a streaming dataframe, to which the same transformations apply as to any other. `writeStream` returns a `DataStreamWriter`, whose methods chain: `format("parquet")`, `option("checkpointLocation", path)`, `trigger(availableNow=True)`, then `start(path)`, which starts the query and returns a `StreamingQuery` right away. Its `awaitTermination()` waits until it has processed everything. The second call resumes from the checkpoint: it only reads the files of March.

```python
def predict_stream() -> None:
  incoming = spark.readStream.schema(...).parquet(...)
  query = (
    reloaded_model.transform(apartments(incoming))
    .drop(...)
    .writeStream.format("parquet")
    .option(...)
    .trigger(...)
    .start("predictions-stream")
  )
  query.awaitTermination()


predict_stream()
print(spark.read.parquet("predictions-stream").count())
deliver(3)
...
```

#### Solution

In [ ]:
# Start from scratch at each run: no sales delivered, no predictions, no checkpoint
for directory in ["incoming", "predictions-stream", "predictions-stream-checkpoint"]:
  shutil.rmtree(directory, ignore_errors=True)
sales_2022 = df.filter(F.year("date") == 2022).cache()


def deliver(month: int) -> None:
  (
    sales_2022.filter(F.month("date") == month)
    .withColumn("month", F.lit(month))
    .write.partitionBy("month")
    .mode("append")
    .parquet("incoming")
  )


deliver(1)
deliver(2)
sales_schema = spark.read.parquet("incoming").schema


def predict_stream() -> None:
  incoming = spark.readStream.schema(sales_schema).parquet("incoming")
  query = (
    reloaded_model.transform(apartments(incoming))
    .drop("features")
    .writeStream.format("parquet")
    .option("checkpointLocation", "predictions-stream-checkpoint")
    .trigger(availableNow=True)
    .start("predictions-stream")
  )
  query.awaitTermination()


predict_stream()
print(spark.read.parquet("predictions-stream").count())
deliver(3)
predict_stream()
print(spark.read.parquet("predictions-stream").count())
# The second pass only read the files of March: the checkpoint keeps the list of the
# files already processed. A real streaming job runs without stopping (a trigger
# every minute, for instance), or is restarted at regular intervals with
# availableNow, which amounts to a batch job that rereads nothing.